# 04 — Hyperparameter Tuning

Optuna search for LightGBM and XGBoost. Uses 3-fold GroupKFold during search (faster), then final evaluation with 5-fold.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import optuna
import warnings
import pickle
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

from pathlib import Path
from sklearn.metrics import roc_auc_score

from src.features import get_feature_cols, TARGET
from src.cv import run_cv, add_target_encoding
from src.models import make_lgb_fn, make_xgb_fn, make_cat_fn
from src.utils import rank_avg, save_submission

cache_dir = Path('..') / 'cache'
train_feat = pd.read_pickle(cache_dir / 'train_feat.pkl')
test_feat  = pd.read_pickle(cache_dir / 'test_feat.pkl')
ext_feat   = pd.read_pickle(cache_dir / 'ext_feat.pkl')

# Re-apply target encoding
train_feat, ext_feat, test_feat = add_target_encoding(train_feat, ext_feat, test_feat)

feature_cols = get_feature_cols(train_feat)
te_cols = [c for c in ['Driver_TE', 'Race_TE', 'Race_Year_TE'] if c in train_feat.columns]
feature_cols = feature_cols + te_cols
print(f'Feature count: {len(feature_cols)}')

## 1. LightGBM Optuna search

In [ ]:
def lgb_objective(trial):
    params = {
        'num_leaves': trial.suggest_int('num_leaves', 63, 511),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.1, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 10.0),
        'subsample_freq': 1,
    }
    fn = make_lgb_fn(params, early_stopping_rounds=50)
    _, fold_aucs = run_cv(train_feat, ext_feat, feature_cols, fn, n_splits=3, verbose=False)
    return np.mean(fold_aucs)

print('Running LightGBM Optuna search (50 trials, 3-fold CV)...')
lgb_study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
lgb_study.optimize(lgb_objective, n_trials=50, show_progress_bar=True)

print(f'\nBest LGB AUC (3-fold): {lgb_study.best_value:.5f}')
print(f'Best params: {lgb_study.best_params}')

In [ ]:
# Full 5-fold CV with best LGB params
print('=== LGB best params — 5-fold CV ===')
best_lgb_fn = make_lgb_fn(lgb_study.best_params)
lgb_oof_tuned, lgb_fold_aucs_tuned = run_cv(train_feat, ext_feat, feature_cols, best_lgb_fn)
print(f'\nLGB tuned OOF AUC: {roc_auc_score(train_feat[TARGET], lgb_oof_tuned):.5f}')

## 2. XGBoost Optuna search

In [ ]:
def xgb_objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 5, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 10.0),
    }
    fn = make_xgb_fn(params, early_stopping_rounds=50)
    _, fold_aucs = run_cv(train_feat, ext_feat, feature_cols, fn, n_splits=3, verbose=False)
    return np.mean(fold_aucs)

print('Running XGBoost Optuna search (30 trials, 3-fold CV)...')
xgb_study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
xgb_study.optimize(xgb_objective, n_trials=30, show_progress_bar=True)

print(f'\nBest XGB AUC (3-fold): {xgb_study.best_value:.5f}')
print(f'Best params: {xgb_study.best_params}')

In [ ]:
print('=== XGB best params — 5-fold CV ===')
best_xgb_fn = make_xgb_fn(xgb_study.best_params)
xgb_oof_tuned, xgb_fold_aucs_tuned = run_cv(train_feat, ext_feat, feature_cols, best_xgb_fn)
print(f'\nXGB tuned OOF AUC: {roc_auc_score(train_feat[TARGET], xgb_oof_tuned):.5f}')

## 3. CatBoost Optuna search

In [ ]:
def cat_objective(trial):
    params = {
        'depth': trial.suggest_int('depth', 5, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.1, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.5, 1.0),
    }
    fn = make_cat_fn(params)
    _, fold_aucs = run_cv(train_feat, ext_feat, feature_cols, fn, n_splits=3, verbose=False)
    return np.mean(fold_aucs)

print('Running CatBoost Optuna search (20 trials, 3-fold CV)...')
cat_study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
cat_study.optimize(cat_objective, n_trials=20, show_progress_bar=True)

print(f'\nBest CAT AUC (3-fold): {cat_study.best_value:.5f}')
print(f'Best params: {cat_study.best_params}')

In [ ]:
print('=== CatBoost best params — 5-fold CV ===')
best_cat_fn = make_cat_fn(cat_study.best_params)
cat_oof_tuned, cat_fold_aucs_tuned = run_cv(train_feat, ext_feat, feature_cols, best_cat_fn)
print(f'\nCAT tuned OOF AUC: {roc_auc_score(train_feat[TARGET], cat_oof_tuned):.5f}')

## 4. Results summary & save

In [ ]:
from src.utils import rank_avg

ensemble_tuned = rank_avg([lgb_oof_tuned, xgb_oof_tuned, cat_oof_tuned])

y_true = train_feat[TARGET].values
print('=== Tuning Results Summary ===')
print(f'LGB tuned : {roc_auc_score(y_true, lgb_oof_tuned):.5f}')
print(f'XGB tuned : {roc_auc_score(y_true, xgb_oof_tuned):.5f}')
print(f'CAT tuned : {roc_auc_score(y_true, cat_oof_tuned):.5f}')
print(f'Rank ens  : {roc_auc_score(y_true, ensemble_tuned):.5f}')

# Save best params
best_params = {
    'lgb': lgb_study.best_params,
    'xgb': xgb_study.best_params,
    'cat': cat_study.best_params,
}
with open(cache_dir / 'best_params.pkl', 'wb') as f:
    pickle.dump(best_params, f)

# Save tuned OOF
oof_tuned = {'lgb': lgb_oof_tuned, 'xgb': xgb_oof_tuned, 'cat': cat_oof_tuned}
with open(cache_dir / 'oof_preds_tuned.pkl', 'wb') as f:
    pickle.dump(oof_tuned, f)

print('\nSaved best params and tuned OOF predictions to cache/')